# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains detailed clinicopathological and molecular records for cancer survivors with second primary colorectal cancer.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show metadata overview
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema defines entities using unique `@id` fields. Here, we list available record sets and their fields using their `@id`s.

In [ ]:
# List available record sets by their @id
# Note: Record sets may need to be retrieved from metadata
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # If record sets are a list, extract @id for each
    if isinstance(metadata.recordSet, list):
        record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in metadata.recordSet]
    else:
        record_sets = [metadata.recordSet['@id']] if isinstance(metadata.recordSet, dict) and '@id' in metadata.recordSet else [metadata.recordSet]

print("Record sets defined in the Croissant schema:")
for idx, rs_id in enumerate(record_sets):
    print(f" {idx+1}. {rs_id}")

# If record sets are empty, attempt to enumerate records distributions
if not record_sets:
    # Try to inspect schema distributions
    if hasattr(metadata, 'distribution'):
        print("\nRecord sets appear empty; attempting to print distribution object(s) @id:")
        distributions = metadata.distribution
        if isinstance(distributions, list):
            for dist in distributions:
                print(f"Distribution @id: {dist['@id'] if isinstance(dist, dict) else dist}")
        else:
            print(f"Distribution @id: {distributions['@id'] if isinstance(distributions, dict) else distributions}")

# Example: Print first 2 records from a first available record set or distribution
record_set_to_use = None
if len(record_sets) > 0:
    record_set_to_use = record_sets[0]
elif hasattr(metadata, 'distribution'):
    # Use distribution @id as a fallback
    dist = metadata.distribution[0] if isinstance(metadata.distribution, list) else metadata.distribution
    record_set_to_use = dist['@id'] if isinstance(dist, dict) and '@id' in dist else dist

if record_set_to_use:
    print(f"\nSample records from record set/distribution: {record_set_to_use}")
    for i, record in enumerate(dataset.records(record_set=record_set_to_use)):
        print(record)
        if i == 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract tabular data into pandas DataFrames using the record set/distribution `@id`.

In [ ]:
# Prepare for data extraction
dataframes = {}

# Use all discovered record_set/distribution IDs
extraction_sets = record_sets if record_sets else []
if not extraction_sets and hasattr(metadata, 'distribution'):
    distributions = metadata.distribution
    if isinstance(distributions, list):
        extraction_sets = [dist['@id'] if isinstance(dist, dict) and '@id' in dist else dist for dist in distributions]
    else:
        extraction_sets = [distributions['@id']] if isinstance(distributions, dict) and '@id' in distributions else [distributions]

# Load data from each record set
for record_set_id in extraction_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from {record_set_id}")
        else:
            print(f"No records found for {record_set_id}")
    except Exception as e:
        print(f"Error loading records from {record_set_id}: {e}")

# Display columns for the primary record set
main_record_set_id = extraction_sets[0] if extraction_sets else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Columns in {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes, using their `@id`s.

In [ ]:
# Select one numeric field for filtering and normalization
current_df = dataframes[main_record_set_id] if main_record_set_id in dataframes else None

# Example candidate numerical field names, adjust based on dataset columns
possible_numeric_fields = ['Age', 'Interval_between_diagnoses', 'Interval_to_second_CRC'] # Use true column @id if available
numeric_field = None
for col in possible_numeric_fields:
    if col in current_df.columns:
        numeric_field = col
        break

if numeric_field:
    threshold = 10
    filtered_df = current_df[current_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping field
    group_field = 'MSI_Status' # Example grouping field, use actual @id if different
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("Numeric field not found; please refer to column list to select a numeric field for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Use fields referenced by their `@id`s.

Below, we provide an example visualization for the numeric field (e.g. age distribution) and relationship between MSI status and age.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if numeric_field and current_df is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(current_df[numeric_field], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Visualize by MSI status if available
    if 'MSI_Status' in current_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x='MSI_Status', y=numeric_field, data=current_df)
        plt.title(f"{numeric_field} by MSI Status")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR^2 dataset and examined available record sets and their fields using their `@id`.
- Extracted tabular clinical and molecular data for cancer survivors with second primary colorectal cancer.
- Performed numerical filtering (e.g. age above threshold) and normalization, and grouped by MSI status where available.
- Visualizations revealed the distribution of numeric fields and their association with molecular phenotypes.

This notebook demonstrated reproducible FAIR data exploration using Croissant and field reference best practices with `mlcroissant`.